# 03 · Preprocessing, processed-data audit and splits

Writes `MyDrive/VeriHGN/processed/ispd15/<prep_hash>/<design>/<sample>/{sample.npz, meta.json}`.
The hash covers the code and the assumptions below, so changing any of them creates a **new** folder and
never overwrites earlier data. Every sample is accounted for as written, already-processed (skipped), or
failed — nothing is dropped silently. Run notebook 02 first in the same session.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
import json, os
from verihgn_repro.data import preprocess as P
from verihgn_repro.data.circuitnet import RawLayout, parse_sample_name, SCOPES
from verihgn_repro.data.splits import make_split_manifests
from verihgn_repro import reporting
from verihgn_repro.colab import write_json
SCOPE = 'official9'
# Unresolved research assumptions — kept at the approved defaults (Stage 5); change only for ablations.
cfg = P.prep_config(knn_k=8, macro_pin_direction='input', with_code_features=True, pin_offset_norm='bbox')
PHASH = P.prep_hash(cfg)
PROCESSED = L.processed_for(PHASH)
print('prep hash:', PHASH, '->', PROCESSED)
print(json.dumps(P.ASSUMPTIONS, indent=1))

In [ ]:
man = P.run(L.local_raw, PROCESSED, scope=SCOPE, cfg=cfg, workers=max(1, (os.cpu_count() or 2) - 1))
disc = reporting.discovery_report(L.local_raw, SCOPE)
prep_report = {'prep_hash': PHASH, 'processed_dir': str(PROCESSED), 'assumptions': P.ASSUMPTIONS,
               'expected': disc['expected_total'], 'requested': man['requested'], 'written': man['written'],
               'skipped_already_processed': man['skipped_already_processed'], 'failed_count': man['failed_count'],
               'failed': man['failed'], 'cell_density_mismatch': man['cell_density_mismatch'], 'seconds': man['seconds']}
write_json(prep_report, L.reports / f'03_preprocessing_{SCOPE}.json')
print(json.dumps({k: v for k, v in prep_report.items() if k != 'assumptions'}, indent=1)[:3000])
assert man['failed_count'] == 0 and not man['cell_density_mismatch'], 'preprocessing failures — inspect before continuing'
assert man['requested'] == disc['expected_total'], 'sample count mismatch'

## Audit every processed sample (feature order, shapes, index ranges, H/V alignment, finiteness)

In [ ]:
expected = [parse_sample_name(s)['sample'] for s in RawLayout(L.local_raw).samples()]
aud = reporting.audit_processed(PROCESSED, SCOPE, expected_samples=expected)
write_json(aud, L.reports / f'03_processed_audit_{SCOPE}.json')
print('audited', aud['audited'], '| ok', aud['ok'], '| failed', len(aud['failed']), '| missing', aud['missing'])
print(json.dumps(aud['per_design'], indent=1))
assert aud['passed'], aud['failed'][:3]

## Splits: official seed-42 and pair-aware (from real netlist signatures)

In [ ]:
splits = make_split_manifests(L.local_raw, SCOPE, L.splits)
srep = reporting.split_report({k: v for k, v in splits.items()}, PROCESSED, SCOPE)
srep['prep_hash'] = PHASH
write_json(srep, L.reports / f'03_splits_{SCOPE}.json')
for name, r in srep.items():
    if name == 'prep_hash': continue
    print(name, r['designs'], r['processed_samples'], 'leak-free:', r['leakage']['leak_free'],
          '| twins across splits:', r['leakage']['netlist_twins_across_splits'])
assert splits['pair_aware']['leakage']['leak_free'] and all(r['all_designs_assigned_once'] for k, r in srep.items() if k != 'prep_hash')